In [1]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb
from merge_tables.db.tables import create_clean_account_name_macro

from pathlib import Path


MERGE_TABLES_DIR = Path('.').parent 
DATA_DIR = MERGE_TABLES_DIR / "data"
OUTPUT_DIR = MERGE_TABLES_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
duck = connect_to_postgres_via_duckdb()
create_clean_account_name_macro(duck)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'
✓ Created clean_account_name macro


In [3]:
duck.sql(
    """
    create or replace table wochenliste as 
    select * replace("ID Nummer"::int64 as "ID Nummer") 
    from read_xlsx('/Users/adrienblanquer/Downloads/2026_KW06_Wochenliste.xlsm', sheet='Kunden', header=True, range='B5:D998')
    """)

In [4]:
duck.sql("from wochenliste")

┌────────────────────────────────────────────────────────────────────┬────────────┬───────────────────┐
│                             Firmenname                             │ ID Nummer  │     Standort      │
│                              varchar                               │   int64    │      varchar      │
├────────────────────────────────────────────────────────────────────┼────────────┼───────────────────┤
│ "Auf der Tenne" e.V.                                               │  130000200 │ Rostock           │
│ &MICA GmbH (ehem. Michels Architekturbüro) - Betriebsstätte Berlin │ 1130100311 │ Berlin            │
│ &MICA GmbH (ehem. Michels Architekturbüro) - Betriebsstätte Köln   │ 1130100312 │ Köln              │
│ 1000hands AG                                                       │  127000001 │ Berlin            │
│ 3D Mapping Solutions GmbH                                          │  130000975 │ München           │
│ 3S Antriebe GmbH                                              

In [12]:
duck.sql(
    """
    select count(*), max(Firmenname)
    from wochenliste
    where length("ID Nummer"::varchar) > 9
    group by left("ID Nummer"::varchar, 9)
    """).show(max_rows=1000)

┌──────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ count_star() │                                             max(Firmenname)                                             │
│    int64     │                                                 varchar                                                 │
├──────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│            2 │ &MICA GmbH (ehem. Michels Architekturbüro) - Betriebsstätte Köln                                        │
│            1 │ LIQID Asset Management GmbH - Standort München                                                          │
│            5 │ Serviceware SE // helpLine GmbH - Betriebsstätte Stuttgart                                              │
│            2 │ Bundesarbeitsgemeinschaft Katholische Jugendsozialarbeot (BAG KJS) e.V. - Betriebsstätte Düsseldorf     │
│            7 │

In [5]:
duck.sql(
    """
    create or replace table wochenliste_multi_firms as 
    with wochenliste_conso as (
    select *, split(Firmenname, '//')[1] as f_split, clean_account_name(f_split) as clean_name
    from wochenliste
    ), multi_firms as (
        select count(*) as nb, clean_name, array_agg("ID Nummer") as ids
        from wochenliste_conso
        group by clean_name
        having nb > 1
    )
    select ids, Firmenname, Standort
    from (select unnest(ids) as ids from multi_firms)
    join wochenliste on ids = "ID Nummer"
    --where Standort <> 'Überregional'

    union all

    select "ID Nummer", Firmenname, Standort
    from wochenliste
    where length(wochenliste."ID Nummer"::varchar) > 9
    and "ID Nummer" not in (select unnest(ids) from multi_firms)
    """
)

# sheet 1

In [71]:
duck.sql(
    """
    select *
    from wochenliste 
    where "ID Nummer" not in (select ids from wochenliste_multi_firms  ) and Standort <> 'Überregional'
    """
).to_csv('output/wochenliste_active_clients_sheet1.csv')

In [72]:
duck.sql(
    """
    create or replace table sheet1 as
    select * from read_csv('output/wochenliste_active_clients_sheet1.csv')
    """
)

In [74]:
duck.sql(
    """
    select *
    from sheet1
    join read_csv('wochenliste_conso.csv')
        on id_easybill::varchar = "ID Nummer"
    where id_easybill is not null and id is not null
    order by easybill_name
    """
).to_csv('output/wochenliste_active_clients_sheet1.csv')

In [11]:
duck.sql(
    """
    create or replace table sheet1 as 
    select
        distinct on("ID Nummer") 
        *
    from read_csv('/Users/adrienblanquer/Downloads/Feuille de calcul sans titre - Single location clients.csv')
    """
)

In [24]:
duck.sql(
    """
    select 
        *, clean_account_name(Firmenname) as clean_name, left ("ID Nummer"::varchar, 9) as easybill_entity_id
    from wochenliste
    where "ID Nummer" not in (select "ID Nummer" from sheet1)
    """
).to_csv('wochenliste_sheets2_3.csv')

In [ ]:
duck.sql(
    """
    select *
    from read_csv('wochenliste_sheets2_3.csv')
    where Standort <> 'Überregional'
        and 
    """
)

┌───────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬───────────────────┬───────────────────────────────────────────┐
│                                        Firmenname                                         │  ID Nummer  │     Standort      │                clean_name                 │
│                                          varchar                                          │    int64    │      varchar      │                  varchar                  │
├───────────────────────────────────────────────────────────────────────────────────────────┼─────────────┼───────────────────┼───────────────────────────────────────────┤
│ &MICA GmbH (ehem. Michels Architekturbüro) - Betriebsstätte Berlin                        │  1130100311 │ Berlin            │ mica                                      │
│ &MICA GmbH (ehem. Michels Architekturbüro) - Betriebsstätte Köln                          │  1130100312 │ Köln              │ mica        

In [140]:
duck.sql(
    """ 
    --create or replace table sorted_multi_firms as
    SELECT 
    --Firmenname,
    clean_name,
    COUNT(DISTINCT easybill_entity_id) AS count_mother_entities,
    COUNT(*) AS count_locations,
    -- Labeling the results based on your logic
    CASE 
        WHEN COUNT(DISTINCT easybill_entity_id) = 1 AND COUNT(*) > 1 
            THEN 'One Mother, Multiple Locations'
        WHEN COUNT(DISTINCT easybill_entity_id) > 1 
            THEN 'Multiple Mothers, Multiple Locations'
        ELSE 'Single Entity, Single Location'
    END AS category,
    -- List the IDs and Locations for reference
    GROUP_CONCAT(DISTINCT easybill_entity_id) AS mother_entity_ids,
    array_agg("ID Nummer") as easybill_ids
    FROM read_csv('wochenliste_sheets2_3.csv')
    where  '//' in Firmenname
    GROUP BY all
    ORDER BY category DESC, count_locations DESC;

    """
)

┌───────────────────────────────────────────────────────────┬───────────────────────┬─────────────────┬──────────────────────────────────────┬───────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                        clean_name                         │ count_mother_entities │ count_locations │               category               │                 mother_entity_ids                 │                                                                                                                easybill_ids                                                                                                                 │
│                          varchar                          │         int64         │      int64      │               varchar         

In [77]:
duck.sql(
    """
    select * 
    from sorted_multi_firms
    where category = 'One Mother, Multiple Locations'
    """
)

┌───────────────────────────────────────────────┬───────────────────────┬─────────────────┬────────────────────────────────┬───────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                  clean_name                   │ count_mother_entities │ count_locations │            category            │ mother_entity_ids │                                                                                                                                            easybill_ids                                                                                                                                            │
│                    varchar                    │         int64         │      int64      │            varchar             │

In [111]:
duck.sql(
    """
    select left("ID Nummer"::varchar, 9) as easybill_entity_id, * 
    from wochenliste
    left join read_csv('wochenliste_conso.csv') as b
        on left(b.id_easybill::varchar, 9) = left("ID Nummer"::varchar, 9) 
    where left("ID Nummer"::varchar, 9) in (select mother_entity_ids from sorted_multi_firms where category = 'One Mother, Multiple Locations')
    ----where left(id_easybill::varchar, 9) in (select mother_entity_ids from sorted_multi_firms where category = 'One Mother, Multiple Locations')
    order by Firmenname

    
    """
).to_csv('output/wochenliste_active_clients_sheet2_2.csv')

In [107]:
duck.sql("select * from read_csv('wochenliste_conso.csv') order by easybill_name")

┌────────────────────────────────────┬─────────────┬─────────────────────────┬──────────────────────────────────────┬───────────┬───────────────────────────┬─────────────────────────────────────────────┬───────────────────────────────────────┬───────────────────────────────────────┬──────────────────────────────────────────────┬────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────┬───────────────────┬────────────────────┐
│                 id                 │ id_easybill │         id_zoho         │             id_medisoft              │ multisite │ should_have_medisoft_firm │                easybill_name                │               zoho_name               │         medisoft_mother_name          │                medisoft_name                 │                       medisoft_path                        │                       medisoft_splited_path                       │     join_type     │        sim         

In [58]:
duck.sql(
    """
    with unnested as (
    select
        unnest(easybill_ids) as easybill_id,
        *
    from sorted_multi_firms a
    )
    select
        *
    from unnested a
    left join read_csv('wochenliste_conso.csv') as b
        on id_easybill = easybill_id
    where clean_name = 'mica'
    order by easybill_name

    """)

┌─────────────┬────────────┬───────────────────────┬─────────────────┬────────────────────────────────┬───────────────────┬──────────────────────────┬────────────────────────────────────┬─────────────┬─────────────────────────┬─────────────┬───────────┬───────────────────────────┬─────────────────────────────────────────────┬────────────┬──────────────────────┬───────────────┬───────────────┬───────────────────────┬───────────────────┬────────┐
│ easybill_id │ clean_name │ count_mother_entities │ count_locations │            category            │ mother_entity_ids │       easybill_ids       │                 id                 │ id_easybill │         id_zoho         │ id_medisoft │ multisite │ should_have_medisoft_firm │                easybill_name                │ zoho_name  │ medisoft_mother_name │ medisoft_name │ medisoft_path │ medisoft_splited_path │     join_type     │  sim   │
│    int64    │  varchar   │         int64         │      int64      │            varchar             

In [61]:
duck.sql("select * from wochenliste")

┌────────────────────────────────────────────────────────────────────┬────────────┬───────────────────┐
│                             Firmenname                             │ ID Nummer  │     Standort      │
│                              varchar                               │   int64    │      varchar      │
├────────────────────────────────────────────────────────────────────┼────────────┼───────────────────┤
│ "Auf der Tenne" e.V.                                               │  130000200 │ Rostock           │
│ &MICA GmbH (ehem. Michels Architekturbüro) - Betriebsstätte Berlin │ 1130100311 │ Berlin            │
│ &MICA GmbH (ehem. Michels Architekturbüro) - Betriebsstätte Köln   │ 1130100312 │ Köln              │
│ 1000hands AG                                                       │  127000001 │ Berlin            │
│ 3D Mapping Solutions GmbH                                          │  130000975 │ München           │
│ 3S Antriebe GmbH                                              

In [ ]:
duck.sql(
    """
    select 
    * from read_csv('wochenliste_conso.csv')
    left join sorted_multi_firms on mother_entity_ids = left(id_easybill::varchar, 9)
    where category = 'One Mother, Multiple Locations'
    --where left(id_easybill::varchar, 9) in (select mother_entity_ids from sorted_multi_firms where category = 'One Mother, Multiple Locations')
    order by easybill_name
    """
)

┌────────────────────────────────────┬─────────────┬─────────────────────────┬──────────────────────────────────────┬───────────┬───────────────────────────┬─────────────────────────────────────────────┬───────────────────────────────────────┬───────────────────────────────────────┬──────────────────────────────────────────────┬────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────┬───────────────────┬────────────────────┐
│                 id                 │ id_easybill │         id_zoho         │             id_medisoft              │ multisite │ should_have_medisoft_firm │                easybill_name                │               zoho_name               │         medisoft_mother_name          │                medisoft_name                 │                       medisoft_path                        │                       medisoft_splited_path                       │     join_type     │        sim         

# sheet 2

In [113]:
duck.sql(
    """
    select *, split(Firmenname, '//')[1] as f_split, clean_account_name(f_split) as clean_name
    from wochenliste_multi_firms
    where length(ids::varchar) > 9
    """
).to_csv('output/wochenliste_multi_firms_sheet2.csv')

In [90]:
duck.sql("from read_csv('wochenliste_conso.csv') as b where left(id_easybill::varchar, 9) = '113010031'")

┌────────────────────────────────────┬─────────────┬─────────────────────────┬─────────────┬───────────┬───────────────────────────┬─────────────────────────────────────────────┬────────────┬──────────────────────┬───────────────┬───────────────┬───────────────────────┬───────────────────┬────────┐
│                 id                 │ id_easybill │         id_zoho         │ id_medisoft │ multisite │ should_have_medisoft_firm │                easybill_name                │ zoho_name  │ medisoft_mother_name │ medisoft_name │ medisoft_path │ medisoft_splited_path │     join_type     │  sim   │
│              varchar               │    int64    │         varchar         │   varchar   │  boolean  │          boolean          │                   varchar                   │  varchar   │       varchar        │    varchar    │    varchar    │        varchar        │      varchar      │ double │
├────────────────────────────────────┼─────────────┼─────────────────────────┼─────────────┼────────

In [115]:
duck.sql(
    """
    select left(ids::varchar, 9) as easybill_entity_id, *
    from read_csv('output/wochenliste_multi_firms_sheet2.csv') as a
    left join read_csv('wochenliste_conso.csv') as b
        on left(b.id_easybill::varchar, 9) = left(a.ids::varchar, 9)
    where id_easybill is not null and id is not null
    order by easybill_name
    """
).to_csv('output/wochenliste_multi_firms_sheet2.csv')
#.show(max_rows=1000)

In [113]:
duck.sql(
    """
    select left("ID Nummer"::varchar, 9) as easybill_entity_id, * 
    from wochenliste
    left join read_csv('wochenliste_conso.csv') as b
        on left(b.id_easybill::varchar, 9) = left("ID Nummer"::varchar, 9) 
    where left("ID Nummer"::varchar, 9) in (select mother_entity_ids from sorted_multi_firms where category = 'One Mother, Multiple Locations')
    ----where left(id_easybill::varchar, 9) in (select mother_entity_ids from sorted_multi_firms where category = 'One Mother, Multiple Locations')
    order by Firmenname

    
    """
).to_csv('output/wochenliste_active_clients_sheet2_2.csv')

# sheet 3

In [119]:
duck.sql(
    """
    select * from wochenliste where Standort = 'Überregional'
    """)

┌─────────────────────────────────────────────────────┬───────────┬──────────────┐
│                     Firmenname                      │ ID Nummer │   Standort   │
│                       varchar                       │   int64   │   varchar    │
├─────────────────────────────────────────────────────┼───────────┼──────────────┤
│ DBB Verlag GmbH                                     │ 104000014 │ Überregional │
│ EJF gemeinnützige AG                                │ 105010000 │ Überregional │
│ HeyJobs GmbH                                        │ 108040004 │ Überregional │
│ Kassenärztliche Vereinigung Nordrhein               │ 130000069 │ Überregional │
│ TBN Transportbeton Nord GmbH & Co. KG               │ 120000056 │ Überregional │
│ UE - University of Europe for Applied Sciences GmbH │ 130001199 │ Überregional │
│ UNI ELEKTRO Fachgroßhandel GmbH & Co. KG            │ 121000003 │ Überregional │
└─────────────────────────────────────────────────────┴───────────┴──────────────┘

In [134]:
duck.sql(
    """
    select * from wochenliste 
    left join read_csv('wochenliste_conso.csv') as b
        on left(b.id_easybill::varchar, 9) = left("ID Nummer"::varchar, 9) 
    where "ID Nummer" in (select unnest(easybill_ids) from sorted_multi_firms where category = 'Multiple Mothers, Multiple Locations')
    --select unnest(easybill_ids), * from sorted_multi_firms where category = 'Multiple Mothers, Multiple Locations' 

    --select left("ID Nummer"::varchar, 9) as easybill_entity_id, * 
    --from wochenliste
    --left join read_csv('wochenliste_conso.csv') as b
    --    on left(b.id_easybill::varchar, 9) = left("ID Nummer"::varchar, 9) 
    --where left("ID Nummer"::varchar, 9) in (select mother_entity_ids from sorted_multi_firms where category = 'Multiple Mothers, Multiple Locations')
    ------where left(id_easybill::varchar, 9) in (select mother_entity_ids from sorted_multi_firms where category = 'One Mother, Multiple Locations')
    --order by Firmenname

    
    """
).to_csv('output/wochenliste_active_clients_sheet3_2.csv')

In [116]:
duck.sql(
    """
    select *, split(Firmenname, '//')[1] as f_split, clean_account_name(f_split) as clean_name
    from wochenliste_multi_firms
    where length(ids::varchar) = 9
    """
)
#.to_csv('output/wochenliste_multi_firms_sheet3.csv')

┌───────────┬────────────────────────────────────────────────────────────────────────────────────────┬───────────────────┬────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────┐
│    ids    │                                       Firmenname                                       │     Standort      │                                        f_split                                         │         clean_name         │
│   int64   │                                        varchar                                         │      varchar      │                                        varchar                                         │          varchar           │
├───────────┼────────────────────────────────────────────────────────────────────────────────────────┼───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┼────────────────────────────┤
│ 130001701 │ Aston Martin Lagonda o

In [ ]:
duck.sql(
    """
    select *
    from read_csv('output/wochenliste_multi_firms_sheet3.csv') as a
    left  join read_csv('wochenliste_conso.csv') as wochen on ids = wochen.id_easybill
    """
).to_csv('output/wochenliste_multi_firms_sheet3.csv')

In [6]:
duck.sql("from read_csv('output/all_easybill_x_zoho_x_medisoft_2.csv') where id_easybill = '119020100'")

┌─────────────┬────────────────────┬───────────────┬───────────────┬─────────────────────┬─────────────┬─────────────────┬───────────────┬─────────────────────┐
│ id_easybill │      id_zoho       │  id_medisoft  │ easybill_name │ easybill_clean_name │  zoho_name  │ zoho_clean_name │ medisoft_name │ medisoft_clean_name │
│   varchar   │       int64        │    varchar    │    varchar    │       varchar       │   varchar   │     varchar     │    varchar    │       varchar       │
├─────────────┼────────────────────┼───────────────┼───────────────┼─────────────────────┼─────────────┼─────────────────┼───────────────┼─────────────────────┤
│ 119020100   │ 386758000010048019 │ 00_92C00MMMT5 │ SUPLEX GmbH   │ suplex              │ SUPLEX GmbH │ suplex          │ Suplex GmbH   │ suplex              │
└─────────────┴────────────────────┴───────────────┴───────────────┴─────────────────────┴─────────────┴─────────────────┴───────────────┴─────────────────────┘

# sheet 4

In [170]:
duck.sql(
    """
    --create or replace table matching_anomalies as 
    select * from wochenliste where "ID Nummer" not in (
        select 
            distinct on("ID Nummer")
            "ID Nummer", 
            
        from read_csv('output/wochenliste_active_clients_sheet3_2.csv')

        union all
        
        select 
        distinct on("ID Nummer")
            "ID Nummer", 
            
        from read_csv('output/wochenliste_active_clients_sheet2_2.csv')

        union all

            select 
        distinct on("ID Nummer")
            "ID Nummer", 
            
        from read_csv('output/wochenliste_active_clients_sheet1.csv')

        union all

        select 
            distinct on("ID Nummer")
            "ID Nummer", 
            
        from read_csv('output/matching_anomalies.csv')
        
    )
    --and Standort <> 'Überregional' and '//' not in Firmenname

    """).to_csv('output/wochenliste_active_clients_sheet4.csv')

In [167]:
duck.sql("from matching_anomalies").to_csv('output/matching_anomalies.csv')

In [163]:
duck.sql(
    """
    select * from read_csv('output/wochenliste_active_clients_sheet1.csv')
    where "ID Nummer" in (
        select 
            distinct on("ID Nummer")
            "ID Nummer", 
            
        from read_csv('output/wochenliste_active_clients_sheet3_2.csv')
    )
    """
)

┌────────────┬───────────┬──────────┬─────────┬─────────────┬─────────┬─────────────┬───────────┬───────────────────────────┬───────────────┬───────────┬──────────────────────┬───────────────┬───────────────┬───────────────────────┬───────────┬────────┐
│ Firmenname │ ID Nummer │ Standort │   id    │ id_easybill │ id_zoho │ id_medisoft │ multisite │ should_have_medisoft_firm │ easybill_name │ zoho_name │ medisoft_mother_name │ medisoft_name │ medisoft_path │ medisoft_splited_path │ join_type │  sim   │
│  varchar   │   int64   │ varchar  │ varchar │    int64    │ varchar │   varchar   │  boolean  │          boolean          │    varchar    │  varchar  │       varchar        │    varchar    │    varchar    │        varchar        │  varchar  │ double │
├────────────┴───────────┴──────────┴─────────┴─────────────┴─────────┴─────────────┴───────────┴───────────────────────────┴───────────────┴───────────┴──────────────────────┴───────────────┴───────────────┴───────────────────────┴──────

# gsheet extension try

In [6]:
duck.sql("""
INSTALL gsheets FROM community;
LOAD gsheets;
""")

In [7]:
duck.sql("""
create secret (type gsheet)
""")


Visit the below URL to authorize DuckDB GSheets
https://accounts.google.com/o/oauth2/v2/auth?client_id=793766532675-rehqgocfn88h0nl88322ht6d1i12kl4e.apps.googleusercontent.com&redirect_uri=https://duckdb-gsheets.com/oauth&response_type=token&scope=https://www.googleapis.com/auth/spreadsheets&state=U3cMFIwsQB
After granting permission, enter the token: 

RuntimeError: Query interrupted